# Comparison of Optimizers under Noisy Data Conditions

Сравнение оптимизаторов **SGD**, **Adam** и **LAMB** на задаче классификации изображений (Animals-10) при различных уровнях шума.

**Два режима эксперимента:**
1. **Training on noise** — каждый оптимизатор обучается и тестируется на одном уровне шума
2. **Robustness** — лучшая модель, обученная на чистых данных, тестируется на всех уровнях шума

Результаты логируются в MLflow, метрики агрегируются по нескольким сидам (mean ± std, CI 95%).

## 1. Установка и настройка окружения

> **Runtime → Change runtime type → GPU** (T4 или лучше). На CPU обучение будет занимать значительно больше времени.

In [ ]:
!git clone https://github.com/irinszn/DL_Optimizers_Experiments.git
%cd DL_Optimizers_Experiments

!pip install -q uv
!uv sync
!pip install -q torch_optimizer mlflow optuna pyngrok scipy

## 2. Google Drive и MLflow

Данные и логи MLflow хранятся на Google Drive. Через ngrok пробрасывается веб-интерфейс MLflow для мониторинга экспериментов.

> **NGROK_AUTHTOKEN** должен быть добавлен в Colab Secrets (`userdata`).

In [ ]:
from google.colab import drive, userdata
from pyngrok import ngrok

drive.mount('/content/drive')

import mlflow
mlflow.set_tracking_uri('file:/content/drive/MyDrive/mlflow')

NGROK_AUTH_TOKEN = userdata.get('NGROK_AUTHTOKEN')
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

import subprocess
subprocess.Popen(['mlflow', 'ui', '--host', '0.0.0.0',
                  '--backend-store-uri', 'file:/content/drive/MyDrive/mlflow',
                  '--port', '5001'])

public_url = ngrok.connect(5001)
print('MLflow UI доступен по адресу:', public_url)

## 3. Загрузка датасета Animals-10 с Kaggle

> Заполните `username` и `key` своими данными из [Kaggle Account → API](https://www.kaggle.com/settings).

In [ ]:
import json
import os

os.makedirs('/root/.kaggle', exist_ok=True)

api_token = {"username": "", "key": ""}
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump(api_token, f)

!chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d alessiocorrado99/animals10
!unzip -q animals10.zip

## 4. Генерация зашумлённых датасетов

Для каждого noise-сценария из конфига генерируется отдельная папка с обработанными изображениями. Генерация **идемпотентна** — уже существующие папки пропускаются.

Используется `generate_datasets_on_drive`: генерация происходит на локальном SSD Colab, затем копируется на Drive (FUSE-запись на Drive напрямую очень медленная).

In [ ]:
from src.data.processing import generate_datasets_on_drive
from src.utils import setup_logging
from run import NOISE_REGISTRY

setup_logging()

CONFIG_PATH = 'configs/example_config.yaml'

generate_datasets_on_drive(config_path=CONFIG_PATH, noise_registry=NOISE_REGISTRY)

## 5. (Опционально) Ручной подбор гиперпараметров

Тюнинг можно включить автоматически: `use_tuner: true` в конфиге — тогда Optuna отработает перед основными экспериментами.

Ниже — пример ручного запуска тюнера для одного оптимизатора (SGD) на одном сценарии (`no_noise`). Найденные параметры можно затем вписать в конфиг в секцию `params`.

In [ ]:
import torch.optim as optim
from src.config import load_config
from src.data.processing import get_dataloaders
from src.models.simple_cnn import SimpleCNN
from src.training.tuner import HyperparameterTuner
from src.utils import SPLIT_RANDOM_STATE
from run import OPTIMIZER_REGISTRY

config = load_config(CONFIG_PATH)

train_loader, val_loader, _ = get_dataloaders(
    preprocessed_root_path=config.data.preprocessed_root_path,
    scenario_folder_template=config.data.scenario_folder_template,
    scenario_name='no_noise',
    random_state=SPLIT_RANDOM_STATE,
    batch_size=config.training.batch_size,
    num_workers=config.data.num_workers,
    pin_memory=config.data.pin_memory,
    subset_size=config.data.debug_subset_size,
)

In [ ]:
opt_config = config.grid_search.optimizers[0]  # SGD

# Params in config but not in search_space are fixed (e.g. nesterov)
tuned_keys = {'lr', 'momentum', 'weight_decay', 'betas'}
fixed_params = {k: v for k, v in opt_config.params.items() if k not in tuned_keys}

sgd_tuner = HyperparameterTuner(
    model_class=SimpleCNN,
    model_params=config.model.params,
    optimizer_name=opt_config.name,
    optimizer_factory=OPTIMIZER_REGISTRY[opt_config.name],
    search_space=opt_config.search_space,
    fixed_params=fixed_params,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs_per_trial=config.training.epochs,
    early_stopping_patience=config.training.early_stopping_patience,
    scheduler_config=config.training.scheduler,
)
sgd_best_params = sgd_tuner.tune(n_trials=40)
print('Best SGD params:', sgd_best_params)

## 6. Основной эксперимент

Запускается сетка **оптимизаторы × noise-сценарии**. Каждая комбинация обучается `num_runs` раз с разными сидами. Для каждого (optimizer, scenario) создаётся parent-run в MLflow с вложенными child-runs для каждого сида.

Если `use_tuner: true`, перед экспериментами автоматически запускается Optuna для сценариев из `tune_scenarios`. Остальные сценарии получают параметры от ближайшего подобранного сценария того же типа шума.

In [ ]:
from run import run_experiments

run_experiments(config_path=CONFIG_PATH)

## 7. Оценка робастности

Для каждого оптимизатора берётся лучшая модель, обученная на чистых данных (`no_noise`), и тестируется на **всех** noise-сценариях без дообучения. Результат — таблица accuracy по сценариям, позволяющая сравнить устойчивость представлений к невиданному шуму.

In [ ]:
from run import run_robustness

run_robustness(config_path=CONFIG_PATH)